# Time-to-first-signal survival review

**Status:** exploratory companion notebook (non-citable)  
**Research question:** B3 — Phase II → III survival probability and time to transition  
([docs/research-questions.md](../../docs/research-questions.md))
**Canonical computation:** `scripts/data/nano_survival_analysis.py` (WS4 / T2)  
**Data as of:** signal files censored at 2026-07-01 (the script's `CUTOFF`)  

Companion view over the survival analysis's canonical inputs, focused on censoring
choices, channel-specific first-signal composition, and cutoff sensitivity. Everything
here is exploratory-tier and non-citable; the published Kaplan–Meier figure remains
generated by the script.

In [ ]:
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
AREA_ID = "nanotechnology"
REPORT_DIR = REPO_ROOT / "data" / "reports" / AREA_ID
RANDOM_SEED = 20260806

## Data contract

- **Population:** area keyword-cohort Phase II awards with an observable Phase II end
  date on or before the cutoff; still-active awards contribute no observation time.
- **Grain:** award for Form D and WS1 evidence; firm for the B82 first-filing channel
  (a firm-level date is applied to each of the firm's awards).
- **Keys:** compound `(award_id, company, award_year)`; award IDs alone are not unique.
- **Inputs:** the three dateable-channel artifacts below. FPDS-coded Phase III and M&A
  signals carry no reliable per-award dates and are structurally excluded, so every
  curve here UNDERSTATES total observable signal.
- **Missingness:** an award with no dateable signal is right-censored at the cutoff,
  not a failure; the undateable channels are absent evidence, not negative evidence.

In [ ]:
ARTIFACTS = {
    "Form D temporal matches": REPORT_DIR / "form_d_post_phase2.csv",
    "WS1 contract evidence": REPORT_DIR / "ws1_contract_evidence.csv",
    "cohort CPC first filings": REPORT_DIR / "cohort_cpc.csv",
}
GENERATORS = {
    "Form D temporal matches": "scripts/data/nano_form_d_temporal.py",
    "WS1 contract evidence": "scripts/data/nano_ws1_contract_evidence.py",
    "cohort CPC first filings": "scripts/data/build_nano_cohort.py",
}
pd.DataFrame(
    [
        {"artifact": name, "path": str(path.relative_to(REPO_ROOT)), "exists": path.exists()}
        for name, path in ARTIFACTS.items()
    ]
)

def load_artifact(name: str) -> pd.DataFrame:
    """Read a canonical CSV artifact, or return an empty frame with a hint."""
    path = ARTIFACTS[name]
    if not path.exists():
        print(
            f"Missing {path.relative_to(REPO_ROOT)} — artifact not present; "
            f"run {GENERATORS[name]} first."
        )
        return pd.DataFrame()
    return pd.read_csv(path, low_memory=False)

## Duration table under an explicit cutoff

Rebuild the per-award duration table the script derives, with the cutoff as a visible
parameter instead of a constant, so censoring choices can be interrogated. The canonical
curve and its published summary statistics still come from `nano_survival_analysis.py`.

In [ ]:
from datetime import date

CUTOFF = date(2026, 7, 1)  # match the script; vary in the sensitivity section below


def parse_date(value) -> date | None:
    try:
        return date.fromisoformat(str(value)[:10])
    except (TypeError, ValueError):
        return None


form_d = load_artifact("Form D temporal matches")
ws1 = load_artifact("WS1 contract evidence")
cpc = load_artifact("cohort CPC first filings")


def build_durations(cutoff: date) -> pd.DataFrame:
    """Per-award (years, event, channel, horizon_years) rows; empty if inputs missing.

    A signal counts as an observed event only when it falls in ``end < date <= cutoff``.
    A signal dated after ``cutoff`` has not been observed as of that cutoff, so the award
    is right-censored at the cutoff (no event). Enforcing the upper bound is what makes
    the cutoff-sensitivity sweep below correct: at an earlier cutoff, a later signal must
    be censored, not counted.
    """
    if form_d.empty or ws1.empty or cpc.empty:
        return pd.DataFrame()
    key_cols = ["award_id", "company", "award_year"]
    ws1_strong = ws1[ws1["evidence_tier"].eq("strong")].copy()
    ws1_strong["ws1_date"] = ws1_strong["first_evidence_date"].map(parse_date)
    ws1_first = ws1_strong.dropna(subset=["ws1_date"]).set_index(key_cols)["ws1_date"].to_dict()
    cpc_first: dict[str, date] = {}
    for company, raw in zip(cpc["company"], cpc.get("cpc_first_b82_filing", ""), strict=False):
        parsed = parse_date(raw)
        if parsed:
            norm = str(company).strip().upper()
            cpc_first[norm] = min(cpc_first.get(norm, parsed), parsed)
    rows = []
    for record in form_d.to_dict("records"):
        end = parse_date(record.get("phase_ii_end_date"))
        if end is None or end > cutoff:
            continue  # still-active awards cannot contribute observation time
        candidates = []
        if str(record.get("form_d_post_p2")) == "True":
            first = parse_date(record.get("form_d_post_p2_first_date"))
            if first is not None and end < first <= cutoff:
                candidates.append(("form_d", first))
        first = ws1_first.get(tuple(record[column] for column in key_cols))
        if first is not None and end < first <= cutoff:
            candidates.append(("ws1_strong", first))
        first = cpc_first.get(str(record["company"]).strip().upper())
        if first is not None and end < first <= cutoff:
            candidates.append(("b82_filing", first))
        horizon_years = (cutoff - end).days / 365.25
        if candidates:
            channel, signal_date = min(candidates, key=lambda item: item[1])
            rows.append({
                "years": (signal_date - end).days / 365.25,
                "event": True,
                "channel": channel,
                "horizon_years": horizon_years,
            })
        else:
            rows.append({
                "years": horizon_years,
                "event": False,
                "channel": "",
                "horizon_years": horizon_years,
            })
    return pd.DataFrame(rows)


durations = build_durations(CUTOFF)
if not durations.empty:
    print(f"{len(durations):,} awards observed; {int(durations['event'].sum()):,} events "
          f"({100 * durations['event'].mean():.1f}%)")
durations.head()

## Censoring sanity check

The cutoff-sensitivity table only holds if a signal dated after the cutoff is right-censored
rather than counted as an event. This cell asserts that invariant against the built frame.

In [ ]:
# An observed event must fall on or before its censoring horizon (cutoff - Phase II end).
# A signal dated after the cutoff is not yet observed and must be censored, never an event.
if durations.empty:
    print("No durations to check — inputs absent.")
else:
    within_horizon = durations["years"] <= durations["horizon_years"] + 1e-9
    assert within_horizon.all(), "an event was counted after the cutoff — censoring is broken"
    censored = ~durations["event"]
    assert (
        (durations.loc[censored, "years"] - durations.loc[censored, "horizon_years"]).abs()
        <= 1e-9
    ).all(), "censored awards must be observed exactly to the horizon"
    early = build_durations(date(2022, 7, 1))
    if not early.empty:
        assert (early["years"] <= early["horizon_years"] + 1e-9).all()
        print(
            f"censoring holds: {int(durations['event'].sum())} events within horizon at {CUTOFF}; "
            f"{int(early['event'].sum())} within horizon at an earlier 2022-07-01 cutoff"
        )
    else:
        print(f"censoring holds: {int(durations['event'].sum())} events within horizon")

## Channel-specific composition

Which channel supplies the *first* signal, and how early. Near-total dominance by one
channel means the curve's shape is really that channel's detection profile.

In [ ]:
if durations.empty:
    channel_profile = pd.DataFrame()
else:
    events = durations[durations["event"]]
    channel_profile = (
        events.groupby("channel")["years"]
        .agg(events="size", median_years="median", p90_years=lambda s: s.quantile(0.9))
        .sort_values("events", ascending=False)
    )
channel_profile

## Cutoff sensitivity

Recompute the matured-award event share under earlier cutoffs. If the share moves
materially, the reported incidence depends on the censoring date, not only on firm
behavior — record that beside any quoted percentage.

In [ ]:
sensitivity_rows = []
for year in (2022, 2023, 2024, 2025, 2026):
    frame = build_durations(date(year, 7, 1))
    if frame.empty:
        continue
    sensitivity_rows.append({
        "cutoff": f"{year}-07-01",
        "awards_observed": len(frame),
        "event_share": round(frame["event"].mean(), 4),
        "median_years_to_event": round(frame.loc[frame["event"], "years"].median(), 2)
        if frame["event"].any()
        else None,
    })
pd.DataFrame(sensitivity_rows)

## Interpretation log

| Observation | Censoring/cutoff dependence | Channel dependence | Defensible statement |
|---|---|---|---|
| _Draft_ | _State how the number moves with the cutoff_ | _Which channel drives it_ | _Descriptive, undateable channels excluded_ |

Published survival figures are regenerated only by `nano_survival_analysis.py` and
checked through the area verifier before any report cites them.